# UK BBS · Download and preprocessing

**Stage 1 of the forecasting workflow** · Python · fixed 2024 data release

Run the cells in order to download the source data, reproduce the supplied R
count aggregation, and add spatial coordinates. This notebook stops before
model fitting or prediction.

**Output:** one row per observed species–square–year, with the maximum visit
count and coordinates in a shared metric CRS.

Source: [BTO / Zenodo record 14161736](https://zenodo.org/records/14161736)
([DOI](https://doi.org/10.5281/zenodo.14161736)), release **2024 (1994–2023 data)**.
The record is deliberately pinned, even if a newer release exists.

## 1. Setup and choices

Follow the setup commands in `README.md`, select the **Python (UK BBS)** kernel,
and run all cells. Keep this notebook in the project folder because it imports
functions from `src/ukbbs/preprocessing.py`.

Edit the settings below if needed. First run: about 352 MB of source downloads.
Later runs verify and reuse cached files. Counts are processed in chunks;
`CHUNKSIZE` controls memory use without changing the result.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Works when Jupyter starts in either the project root or notebooks/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "ukbbs" / "preprocessing.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the extracted ukbbs-workflow folder.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ukbbs.preprocessing import (
    SOURCE_URL, COUNT_COLUMN, SECTIONS, TARGET_CRS,
    download_release, preprocess_counts, survey_coverage,
    coordinate_lookup, attach_coordinates, write_manifest,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
EXCLUDED_YEARS = (2001, 2020)
CHUNKSIZE = 500_000
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT.name)
print("Excluded years:", EXCLUDED_YEARS)
print("Output coordinates:", TARGET_CRS, "(metres)")

Project: ukbbs-workflow
Excluded years: (2001, 2020)
Output coordinates: EPSG:3035 (metres)


## 2. Download and verify the fixed release

Files are streamed to disk and checked against the publisher's MD5 checksums.
Interrupted downloads do not replace completed files. A damaged cached file
raises an error; remove that file and rerun this cell to download it again.

Alongside the bird counts, download the visit table, coordinate lookup, species
lookup, and metadata. The visit table is needed to describe sampling coverage;
read `0_Metadata.pdf` before interpreting or extending this workflow.

In [2]:
files = download_release(RAW_DIR)
display(pd.DataFrame([
    {"file": name, "size_MB": round(path.stat().st_size / 1_000_000, 3)}
    for name, path in files.items()
]))

Verified cached BBS_bird_dataset.csv
Verified cached BBS_visit_dataset.csv
Verified cached grid_square_coordinates_lookup.csv
Verified cached species_lookup.csv
Verified cached 0_Metadata.pdf


,file,size_MB
0,BBS_bird_dataset.csv,354.810
1,BBS_visit_dataset.csv,11.772
2,grid_square_coordinates_lookup.csv,0.358
3,species_lookup.csv,0.009
4,0_Metadata.pdf,0.206


## 3. Inspect the input

The source row identifies **year, square, visit, species and distance band**.
The ten `section_` columns hold counts along the transect sections.

This workflow includes every band as supplied, including **F (flight only)**. Counts with missing distance-band labels also remain included
and are reported in the diagnostics. No species filter or visit rule is changed.

In [3]:
raw_preview = pd.read_csv(files["BBS_bird_dataset.csv"], nrows=5)
display(raw_preview)
print("Section columns:", SECTIONS)

display(pd.read_csv(files["species_lookup.csv"], keep_default_na=False).head())

,year,square,visit,species_code,distance_band,section_1,section_2,section_3,section_4,section_5,section_6,section_7,section_8,section_9,section_10
0,1994,HU3656,E,CU,3,1,0,0,0,0,0,0,0,0,0
1,1994,HU3656,E,GP,3,0,0,0,0,0,0,1,0,0,0
2,1994,HU3656,E,MP,3,0,0,1,1,0,0,0,0,0,0
3,1994,HU3656,E,MP,2,0,0,0,0,0,0,0,0,0,1
4,1994,HU3656,E,S.,3,0,0,0,0,1,0,0,0,0,0


Section columns: ['section_1', 'section_2', 'section_3', 'section_4', 'section_5', 'section_6', 'section_7', 'section_8', 'section_9', 'section_10']


,species_code,English_name,scientific_name
0,AB,Sabine's Gull,Xema sabini
1,AC,Parasitic Jaeger,Stercorarius parasiticus
2,AE,Arctic Tern,Sterna paradisaea
3,AS,Black Swan,Cygnus atratus
4,AX,Alexandrine Parakeet,Psittacula eupatria


## 4. Aggregate counts and exclude years

For each source row, sum sections. Then, for each species, square and year:

1. Sum row totals across all distance bands **within each visit**.
2. Keep the **maximum visit total** (or the one available visit).
3. Remove years 2001 and 2020.

For example, an early visit with band totals 5, 4 and 5 gives **14**. If the late
visit totals 12, the annual response is **14**.

Missing section values contribute zero. An all-missing section row therefore
also totals zero, matching `rowSums(..., na.rm=TRUE)` in the original R code.
The diagnostics below report missing values so this choice stays visible.

The chunked implementation re-sums visit totals across chunks **before** taking
annual maxima. Source rows are summed as supplied; no count rows are deduplicated.

In [4]:
model_data, diagnostics = preprocess_counts(
    files["BBS_bird_dataset.csv"],
    excluded_years=EXCLUDED_YEARS,
    chunksize=CHUNKSIZE,
)
display(model_data.head(10))
display(pd.Series(diagnostics, name="value").to_frame())

,OS_GRID_REF,species,max_total_count_over_all_distance_bands_for_max_visit,year
0,HU3656,CU,1,1994
1,HU3656,GP,1,1994
2,HU3656,MP,5,1994
3,HU3656,S.,6,1994
4,HU3656,W.,1,1994
5,HU4353,AC,2,1994
6,HU4353,CM,13,1994
7,HU4353,CU,12,1994
8,HU4353,F.,1,1994
9,HU4353,GB,1,1994


,value
raw_rows,7369495
excluded_rows,144269
missing_section_cells,860578
missing_distance_band_rows,10434
all_missing_section_rows,0
distance_bands,"[(missing), 1, 2, 3, F]"
output_rows,2719011
square_species_visits,4283712
unique_squares,7070
unique_species,216


In [5]:
# Structural checks before adding coordinates.
assert list(model_data.columns) == ["OS_GRID_REF", "species", COUNT_COLUMN, "year"]
assert not model_data.duplicated(["OS_GRID_REF", "species", "year"]).any()
assert not model_data.year.isin(EXCLUDED_YEARS).any()
assert model_data[COUNT_COLUMN].ge(0).all()
assert model_data[COUNT_COLUMN].mod(1).eq(0).all()
assert model_data.notna().all().all()
print(f"Prepared {len(model_data):,} observed species-square-year rows.")
print(f"Count-table squares: {model_data.OS_GRID_REF.nunique():,}")
print(f"Species codes: {model_data.species.nunique():,}")
print(f"Year range: {model_data.year.min()}–{model_data.year.max()}")

Prepared 2,719,011 observed species-square-year rows.
Count-table squares: 7,070
Species codes: 216
Year range: 1994–2024


## 5. Check survey coverage

Use **`BBS_visit_dataset.csv`** to identify surveyed square-years. Coverage
computed only from bird detections can understate survey effort.

“Sampled every retained year” means every calendar year in the visit table's
span except the explicitly excluded years. The annual table below compares
sampled squares with squares represented in the processed bird table.

**No zero imputation happens here.** A missing species–square–year row may mean
a non-detection or no survey. Keep the sampled-square-year table for the later
modelling stage, when that distinction and species-specific site selection
will be handled.

In [6]:
sampled_square_years, square_coverage, retained_years = survey_coverage(
    files["BBS_visit_dataset.csv"], excluded_years=EXCLUDED_YEARS,
)

# Every observed count must correspond to a surveyed square-year.
observed_square_years = model_data[["OS_GRID_REF", "year"]].drop_duplicates()
visit_check = observed_square_years.merge(
    sampled_square_years, on=["OS_GRID_REF", "year"],
    how="left", validate="one_to_one", indicator=True,
)
assert visit_check._merge.eq("both").all(), "Count records without matching survey visits"

print("Retained calendar years:", len(retained_years))
print("Survey-table squares:", f"{len(square_coverage):,}")
print("Squares sampled in every retained year:",
      f"{square_coverage.sampled_every_retained_year.sum():,}")
display(square_coverage.n_years_sampled.describe().to_frame())
display(square_coverage.head())

Retained calendar years: 29
Survey-table squares: 7,070
Squares sampled in every retained year: 168


,n_years_sampled
count,7070.000000
mean,13.120085
std,7.987073
min,1.000000
25%,6.000000
50%,12.000000
75%,19.000000
max,29.000000


,OS_GRID_REF,n_years_sampled,sampled_every_retained_year
0,HP6408,3,False
1,HU2078,3,False
2,HU3160,1,False
3,HU3177,2,False
4,HU3274,1,False


In [7]:
annual_coverage = pd.DataFrame(index=pd.Index(retained_years, name="year"))
annual_coverage["sampled_squares"] = sampled_square_years.groupby("year").OS_GRID_REF.nunique()
annual_coverage["squares_with_count_rows"] = observed_square_years.groupby("year").OS_GRID_REF.nunique()
annual_coverage = annual_coverage.fillna(0).astype(int)
display(annual_coverage)

,sampled_squares,squares_with_count_rows
year,,
1994,1568,1568
1995,1746,1746
1996,1919,1919
1997,2194,2194
1998,2310,2310
1999,2395,2395
2000,2301,2301
2002,2204,2204
2003,2324,2324


## 6. Attach coordinates in a common CRS

Your supplied coordinate script is retained in `reference/original_coordinates.py`.
This workflow makes one deliberate change: **use the publisher's coordinate
lookup rather than manually parsing grid references**. The lookup includes
Great Britain, Northern Ireland, the Isle of Man and the Channel Islands.

The metadata describes `ETRS89Long` and `ETRS89Lat` as square centroids. Transform
these from **ETRS89 (EPSG:4258)** to **ETRS89-extended / LAEA Europe (EPSG:3035)**,
with longitude first. Do not add another half-square offset.

All `easting` and `northing` outputs are in this common CRS, in metres. Their
`_km` counterparts divide both axes by 1,000 and round to four decimal places.
`grid_system` describes the original reference notation, not the output CRS.
The source square size is 1,000 m; coordinates do not imply finer survey precision.

The lookup is transformed once per square. A validated many-to-one join prevents
row multiplication and fails if any count location is missing.

In [8]:
all_locations = coordinate_lookup(files["grid_square_coordinates_lookup.csv"])
locations = all_locations.loc[
    all_locations.OS_GRID_REF.isin(model_data.OS_GRID_REF)
].copy()
model_data_with_coordinates = attach_coordinates(model_data, locations)

assert len(model_data_with_coordinates) == len(model_data)
assert np.isfinite(model_data_with_coordinates[["easting", "northing"]]).all().all()
assert model_data_with_coordinates.coordinate_crs.eq(TARGET_CRS).all()

display(locations.groupby("grid_system").size().rename("n_squares").to_frame())
display(model_data_with_coordinates[[
    "OS_GRID_REF", "species", "year", COUNT_COLUMN,
    "grid_system", "easting", "northing", "easting_km", "northing_km",
]].head(10))
print("All rows have finite coordinates; count-table row count is unchanged.")

,n_squares
grid_system,
British National Grid,6796
Channel Islands grid,41
Irish Grid,233


,OS_GRID_REF,species,year,max_total_count_over_all_distance_bands_for_max_visit,grid_system,easting,northing,easting_km,northing_km
0,HU3656,CU,1994,1,British National Grid,3.695229e+06,4.182562e+06,3695.2291,4182.5619
1,HU3656,GP,1994,1,British National Grid,3.695229e+06,4.182562e+06,3695.2291,4182.5619
2,HU3656,MP,1994,5,British National Grid,3.695229e+06,4.182562e+06,3695.2291,4182.5619
3,HU3656,S.,1994,6,British National Grid,3.695229e+06,4.182562e+06,3695.2291,4182.5619
4,HU3656,W.,1994,1,British National Grid,3.695229e+06,4.182562e+06,3695.2291,4182.5619
5,HU4353,AC,1994,2,British National Grid,3.701600e+06,4.178421e+06,3701.6000,4178.4206
6,HU4353,CM,1994,13,British National Grid,3.701600e+06,4.178421e+06,3701.6000,4178.4206
7,HU4353,CU,1994,12,British National Grid,3.701600e+06,4.178421e+06,3701.6000,4178.4206
8,HU4353,F.,1994,1,British National Grid,3.701600e+06,4.178421e+06,3701.6000,4178.4206
9,HU4353,GB,1994,1,British National Grid,3.701600e+06,4.178421e+06,3701.6000,4178.4206


All rows have finite coordinates; count-table row count is unchanged.


## 7. Save outputs and run provenance

Save both the original four-column table and its coordinate-enriched version.
The extra location and coverage tables will be useful for modelling. A JSON
manifest records source checksums, processing choices and software versions.

Raw and generated data are ignored by Git. Collaborators regenerate them from
this notebook using the same fixed source release.

In [9]:
outputs = {
    "ukbbs_model_data.csv": model_data,
    "ukbbs_model_data_with_coordinates.csv": model_data_with_coordinates,
    "locations.csv": locations,
    "sampled_square_years.csv": sampled_square_years,
    "square_coverage.csv": square_coverage,
}
for filename, table in outputs.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"Saved {filename}: {len(table):,} rows")

diagnostics["retained_years"] = retained_years
diagnostics["survey_squares"] = len(square_coverage)
diagnostics["squares_sampled_every_retained_year"] = int(
    square_coverage.sampled_every_retained_year.sum()
)
write_manifest(
    OUTPUT_DIR / "preprocessing_manifest.json", files, diagnostics,
    excluded_years=EXCLUDED_YEARS,
)
print("Saved preprocessing_manifest.json")

Saved ukbbs_model_data.csv: 2,719,011 rows
Saved ukbbs_model_data_with_coordinates.csv: 2,719,011 rows
Saved locations.csv: 7,070 rows
Saved sampled_square_years.csv: 92,759 rows
Saved square_coverage.csv: 7,070 rows
Saved preprocessing_manifest.json


In [10]:
# import pandas as pd

# original = pd.read_csv(
#     "/mnt/bulk/ts-predict/inputs/ukbbs_model_data.csv",
#     keep_default_na=False,
# )
# notebook = pd.read_csv(
#     "/mnt/bulk/ukbbs-workflow/data/processed/ukbbs_model_data.csv",
#     keep_default_na=False,
# )

# keys = ["year", "OS_GRID_REF", "species"]

# for name, df in [("Original", original), ("Notebook", notebook)]:
#     print(f"{name}: {len(df):,} rows")
#     assert not df.duplicated(keys).any(), f"{name} has duplicate keys"

# pd.testing.assert_frame_equal(
#     original.sort_values(keys).reset_index(drop=True),
#     notebook.sort_values(keys).reset_index(drop=True),
#     check_dtype=False,
#     check_exact=True,
#     check_like=True,
# )

# print("PASS: all rows and values match.")

## Next: review before modelling

The outputs are ready for the next workflow stage. Decisions for that stage:

- Use survey visits to distinguish non-detection zeros from missing surveys.
- Choose species, minimum site coverage, and treatment of sites with one year.
- Decide how parent/adjacent squares and survey stratification enter the model.
- Define a chronological training/validation split and forecast horizon.

This notebook retains all supplied species and bands and applies no complete
panel construction, model fitting, or predictions. It reproduces your count
processing rather than the complete official BBS trend procedure.

**Source guidance:** read the downloaded `0_Metadata.pdf` and the associated
Massimino et al. article. The metadata labels the data CC BY-NC and explains
BTO/JNCC/RSPB attribution. See [the source record](https://zenodo.org/records/14161736).
For coordinate-transform API details, see the
[pyproj Transformer documentation](https://pyproj4.github.io/pyproj/stable/api/transformer.html).